# 07c · 第七讲基础巩固习题：PyTorch 神经网络与显微图像卷积

本习题集围绕 `07a`（人工神经元、Autograd 自动微分、多层感知机与 MC Dropout 不确定性）和 `07b`（二维离散卷积、池化与金相微结构 CNN）设计了 **8 道由浅入深的基础练习**。

- 每道题均提供完整的数据或张量初始化代码，只需在 `# TODO` 位置补全核心语句即可直接运行验证。
- 完整参考答案见配套的 `solutions/07_exercises_solutions.ipynb`。

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
torch.manual_seed(42)
np.random.seed(42)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 第一部分：PyTorch 张量、自动微分与多层感知机

### 习题 1：张量行标准化与矩阵乘法

给定一个表示 4 个无机材料样品、每个样品 3 个物理描述符的二维浮点张量 `X`（形状 `(4, 3)`）以及权重向量 `w`（形状 `(3, 1)`）：
1. 计算 `X` **每一列（每个特征）**的均值 `col_mean` 和标准差 `col_std`（提示：`X.mean(dim=0)` 与 `X.std(dim=0)`），并将 `X` 标准化为 `X_norm = (X - col_mean) / col_std`；
2. 使用矩阵乘法运算符 `@` 计算线性预测输出 `y_lin = X_norm @ w`。

In [ ]:
X = torch.tensor([
    [300.0, 1.5, 12.0],
    [450.0, 2.1, 18.0],
    [600.0, 3.0, 24.0],
    [750.0, 3.6, 30.0],
], dtype=torch.float32)
w = torch.tensor([[0.8], [-0.5], [1.2]], dtype=torch.float32)

# TODO: 按列标准化 X 并计算矩阵乘法 X_norm @ w
col_mean = None
col_std = None
X_norm = None
y_lin = None

# print("Normalized X:\n", X_norm)
# print("Linear output y_lin:\n", y_lin)

---
### 习题 2：用 Autograd 求多元自由能函数的偏导数

假设某二元固溶体的简化自由能曲面上一点由温度标量 $T = 3.0$ 和成分标量 $c = 2.0$ 决定，函数表达式为：

$$G(T, c) = 2 c^3 - 3 T c^2 + T^2$$

请：
1. 创建两个开启自动求导的标量张量 `T = torch.tensor(3.0, requires_grad=True)` 与 `c = torch.tensor(2.0, requires_grad=True)`；
2. 计算 `G` 并调用 `G.backward()`；
3. 打印自动求导得到的 $\partial G / \partial T$ 与 $\partial G / \partial c$，并与手算结果（$\partial G / \partial T = -3c^2 + 2T = -6$，$\partial G / \partial c = 6c^2 - 6Tc = -12$）核对。

In [ ]:
# TODO: 定义 requires_grad=True 的张量 T 与 c，计算 G 并反向传播求偏导
T = None
c = None
G = None

# print("G(3, 2) =", G.item())
# print("dG/dT =", T.grad.item(), "| dG/dc =", c.grad.item())

---
### 习题 3：手算与验证多层感知机的参数总数

考虑一个用于预测晶体形成能的三层感知机：
- 输入特征维度为 `10`；
- 第一隐藏层有 `32` 个神经元（带偏置），接 `nn.ReLU()`；
- 第二隐藏层有 `16` 个神经元（带偏置），接 `nn.ReLU()`；
- 输出层有 `1` 个神经元（带偏置）。

请：
1. 先用算术表达式手算总参数量 `manual_count`；
2. 用 `nn.Sequential` 搭建该网络 `net`，并用 `sum(p.numel() for p in net.parameters())` 验证两者完全相等。

In [ ]:
# TODO: 手算参数总数并构建 nn.Sequential 验证
manual_count = None
net = None

# print("Manual count: ", manual_count)
# print("PyTorch count:", sum(p.numel() for p in net.parameters()))

---
### 习题 4：编写自定义 `nn.Module` 并用 Adam 拟合非线性曲线

下方生成了一组带噪声的一维非线性实验数据 $y = \cos(2x) + 0.3x + \epsilon$（共 120 个样本）。请：
1. 完成继承自 `nn.Module` 的类 `SimpleMLP`（结构：`1 -> 32 -> ReLU -> 32 -> ReLU -> 1`）；
2. 使用 `torch.optim.Adam(model.parameters(), lr=1e-2)` 和 `nn.MSELoss()` 训练 200 轮；
3. 绘制训练后的拟合曲线与原始散点图。

In [ ]:
torch.manual_seed(0)
x_data = torch.linspace(-2.5, 2.5, 120).unsqueeze(1)  # (120, 1)
y_data = (torch.cos(2 * x_data) + 0.3 * x_data + 0.08 * torch.randn_like(x_data)).squeeze(1)

class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: 定义 1 -> 32 -> ReLU -> 32 -> ReLU -> 1 的网络层
        pass

    def forward(self, x):
        # TODO: 前向传播并将输出形状压缩为 (batch,)
        pass

# TODO: 实例化模型并用 Adam (lr=1e-2) 训练 200 轮


---
### 习题 5：实现 MC Dropout 推理并比较内插与外推不确定性

下面为您提供了一个带有 `nn.Dropout(0.25)` 的预训练网络 `dropout_net`（在区间 $x \in [-2, 2]$ 的数据上训练完成）。
请在测试点 $x_{\text{in}} = 0.5$（训练区内插点）与 $x_{\text{out}} = 5.0$（远离训练区的外推点）上：
1. 保持 `dropout_net.train()` 开启，在 `torch.no_grad()` 下重复前向传播 $T = 100$ 次；
2. 分别打印两个测试点处预测值的均值与标准差。

In [ ]:
torch.manual_seed(42)
dropout_net = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.25),
    nn.Linear(64, 1),
)
# 在 [-2, 2] 上快速训练 150 步
x_tr_mc = torch.linspace(-2.0, 2.0, 80).unsqueeze(1)
y_tr_mc = torch.sin(x_tr_mc)
opt_mc = torch.optim.Adam(dropout_net.parameters(), lr=1e-2)
for _ in range(150):
    dropout_net.train()
    loss = F.mse_loss(dropout_net(x_tr_mc), y_tr_mc)
    opt_mc.zero_grad(); loss.backward(); opt_mc.step()

x_probe = torch.tensor([[0.5], [5.0]])

# TODO: 保持 dropout_net.train()，对 x_probe 采样 100 次，计算均值与标准差
mc_samples = None


---
## 第二部分：二维离散卷积与金相微结构 CNN

### 习题 6：用水平 Sobel 核提取层片状组织的水平相界

在 `07b` 正文中我们使用了检测垂直边缘的 Sobel 核 `k_sobel_x`。现在给定一张含有**水平层片条纹**的 $32\times 32$ 合成金相图 `horiz_img`：
1. 构造检测**水平边缘（垂直方向灰度梯度 $\partial I / \partial y$）**的 $3\times 3$ Sobel 卷积核：
   $$K_y = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$
   并将其形状调整为 `(1, 1, 3, 3)`；
2. 调用 `F.conv2d(horiz_img, k_sobel_y, padding=1)` 计算边缘响应图并可视化。

In [ ]:
yy, xx = np.mgrid[0:32, 0:32].astype(np.float32)
horiz_np = ((np.sin(2 * np.pi * yy / 8.0) > 0).astype(np.float32) * 0.7 + 0.15)
horiz_img = torch.tensor(horiz_np)[None, None, :, :]  # (1, 1, 32, 32)

# TODO: 构造形状为 (1, 1, 3, 3) 的水平 Sobel 核 k_sobel_y，并调用 F.conv2d
k_sobel_y = None
edge_y = None


---
### 习题 7：推导并验证卷积 + 池化流水线的输出形状

给定输入图像批次 `x_img = torch.zeros(4, 1, 32, 32)`（4 张单通道 $32\times 32$ 图像），依次经过：
1. `nn.Conv2d(1, 8, kernel_size=3, padding=1)` + `nn.ReLU()`
2. `nn.MaxPool2d(kernel_size=2, stride=2)`
3. `nn.Conv2d(8, 16, kernel_size=3, padding=0)`（注意此处 `padding=0`！）+ `nn.ReLU()`
4. `nn.MaxPool2d(kernel_size=2, stride=2)`

请先手算第 4 步输出张量的形状 `(Batch, Channels, H_out, W_out)`，再用 `nn.Sequential` 搭建该特征提取器并打印输出形状。

In [ ]:
x_img = torch.zeros(4, 1, 32, 32)

# TODO: 构建上述 4 步卷积池化序列并打印输出形状
feature_extractor = None


---
### 习题 8：训练轻量级 CNN 区分“水平层片”与“垂直层片”微结构

下方代码生成了 160 张 $24\times 24$ 的含噪条纹微结构图：类别 `0` 为水平取向层片，类别 `1` 为垂直取向层片（前 120 张为训练集，后 40 张为测试集）。
请搭建一个包含 `Conv2d(1, 8, kernel_size=3, padding=1) -> ReLU -> AdaptiveAvgPool2d((1, 1)) -> Flatten -> Linear(8, 2)` 的极简 CNN，使用 `nn.CrossEntropyLoss()` 训练 25 轮，并打印测试集分类准确率。

In [ ]:
rng = np.random.RandomState(42)
yy, xx = np.mgrid[0:24, 0:24].astype(np.float32) / 24.0
imgs_8, lbls_8 = [], []
for i in range(160):
    cls = i % 2
    freq = rng.uniform(3.5, 6.5)
    phase = rng.uniform(0, 2 * np.pi)
    coord = yy if cls == 0 else xx
    clean = (np.sin(2 * np.pi * freq * coord + phase) > 0).astype(np.float32) * 0.7 + 0.15
    noisy = np.clip(clean + rng.normal(0, 0.08, (24, 24)).astype(np.float32), 0.0, 1.0)
    imgs_8.append(noisy)
    lbls_8.append(cls)

X8 = torch.tensor(np.array(imgs_8)[:, None, :, :])
y8 = torch.tensor(lbls_8, dtype=torch.long)
X8_tr, X8_te = X8[:120], X8[120:]
y8_tr, y8_te = y8[:120], y8[120:]

# TODO: 搭建极简 CNN，在 (X8_tr, y8_tr) 上训练 25 轮，并计算 (X8_te, y8_te) 上的测试准确率
mini_cnn = None
